# EDA: `combined_2221-2228.csv`

This notebook performs a comprehensive exploratory data analysis (EDA) on the uploaded CSV `/mnt/data/combined_2221-2228.csv`.

**Contents**
- Robust CSV loading (encoding fallback)
- Head, shape, and `info()`
- Column-level summaries (dtype, unique counts, missing %)
- Missing values and duplicates
- Detection of null-like placeholders ("NA", "N/A", "-")
- Mixed-type detection (numeric-looking strings in object columns)
- Date parsing heuristics
- Numeric diagnostics with histograms & boxplots (matplotlib only)
- Categorical diagnostics
- Correlation matrix (numeric columns)
- Consolidated issues report and example cleaning snippets

Run each cell sequentially.


In [ ]:
# CELL 1 - Setup & configuration
import os, io, sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
# Use Agg backend in non-interactive contexts (safe)
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datetime import datetime
from zipfile import ZipFile
from IPython.display import display, HTML

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 400)

# ===== CONFIG: update these =====
CSV_PATH = Path(r"TMParseOCR\combined_data\output_single_csv\combined_2221-2228.csv") # path to your combined CSV
OUT_DIR = Path(r"TMParseOCR\eda_reports")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUT_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

MAX_DISPLAY_ROWS = 200
NULL_LIKE_PATTERNS = ['na','n/a','none','-','?','null','nil','unknown']
HTML_REPORT_PATH = OUT_DIR / "EDA_report.html"
ZIP_PATH = OUT_DIR.with_suffix(".zip")
# =================================


In [6]:
# CELL 2 - Helpers
def safe_display_df(df, title=None, max_rows=MAX_DISPLAY_ROWS):
    if title:
        print(f"--- {title} ---")
    if isinstance(df, pd.DataFrame) and len(df) > max_rows:
        display(df.head(max_rows))
        print(f"... (showing top {max_rows} rows out of {len(df)})")
    else:
        display(df)

def timestamp():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def save_fig(fig, fname, dpi=150):
    p = PLOTS_DIR / fname
    fig.tight_layout()
    fig.savefig(p, dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    return p

def df_to_html_table(df, title=None, max_rows=500):
    if df is None or df.empty:
        return f"<h3>{title or ''}</h3><p><em>Empty</em></p>"
    display_df = df if len(df) <= max_rows else df.head(max_rows)
    html = "<h3>{}</h3>".format(title or "")
    html += display_df.to_html(classes='table table-striped', border=0, index=False, escape=False)
    if len(df) > max_rows:
        html += f"<p>... showing top {max_rows} rows (total {len(df)})</p>"
    return html


In [7]:
 # CELL 3 - Load CSV
assert CSV_PATH.exists(), f"CSV not found at {CSV_PATH}"
def load_csv(path):
    try:
        df = pd.read_csv(path)
        print(f"Loaded with default encoding. Shape: {df.shape}")
        return df
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding='latin1')
        print(f"Loaded with latin1 encoding. Shape: {df.shape}")
        return df
    except Exception as e:
        raise RuntimeError(f"Failed to load CSV: {e}")

df = load_csv(CSV_PATH)
print("Head:")
safe_display_df(df.head(10), "Head (first 10 rows)")
print("Shape:", df.shape)
print("Dtypes:")
display(df.dtypes)


Loaded with default encoding. Shape: (18514, 17)
Head:
--- Head (first 10 rows) ---


,journal_no,registration_date,nice_classes,trademark_text,trademark_text_devanagari,international_registration_no,image_url,application_number,filing_date,owner_name,owner_name_devanagari,owner_address,used_since,status,registration_city,goods_services_description,business_type_category
0,2221.0,11-08-2025,1,????? ????,????? ????,NaN,logos/4061826.png,4061826.0,19-01-2019,Farm Bio-Tech Pvt Ltd,फार्म बायो-टेक पवत ल्टड,"B 607, Titanium Square, Near Thaltej Cross Road, Thaltej, Ahmedabad 380054, Gujarat, India. A Company Incorporated under the Indian Company's Act.",18-06-2011,registered,Ahmedabad,"Agricultural chemicals, except fungicides, weedkillers, herbicides, insecticides and parasiticides, Manure for agriculture, Chemical fertilizers, Fertilizers, Compost (Fertilizer), Bio-fertilizer, Phosphate fertilizers, Superphosphate fertilizers, Liquid fertilizer, compound fertilizers, complex fertilizers, manganese fertilizers, Urea fertilizers, Organic fertilizers, Land fertilizers. F Colo...",goods
1,2221.0,11-08-2025,1,Starter Plus Starter Plus Nutri Starter Plus Upt Colar Godr,स्टार्टर प्लस स्टार्टर प्लस नोटरी स्टार्टर प्लस ुप्त कर गोदर,NaN,logos/5805302.png,5805302.0,13-02-2023,Green Agrevolution Private Limited,ग्रीन अगरेवोलुशन प्राइवेट लिमिटेड,"901, 9th Floor, BPTP Park Centra, Sector 30, Gurgaon, 122001 A company incorporated",NaN,pending,Delhi,Biofertilizers and Non-chemical biofertilizers. Condition: the mark shall be used as a whole.,goods
2,2221.0,11-08-2025,1,Evolution,ेवोलुशन,NaN,logos/5958142.png,5958142.0,30-05-2023,Gaurav Murjani,गौरव मूरजानी,"118/105 (ID, BAMBA ROAD, GUMTI NO.5, KANPUR, UTTAR PRADESH-208012 INDIVIDUAL",NaN,pending,Delhi,"CHEMICALS FOR USE IN INDUSTRY, SCIENCE AND PHOTOGRAPHY, AS WELL AS IN AGRICULTURE, HORTICULTURE AND FORESTRY; UNPROCESSED ARTIFICIAL RESINS, UNPROCESSED PLASTICS; FIRE EXTINGUISHING AND FIRE PREVENTION COMPOSITIONS; TEMPERING AND SOLDERING PREPARATIONS; SUBSTANCES FOR TANNING ANIMAL SKINS AND HIDES; ADHESIVES FOR USE IN INDUSTRY; PUTTIES AND OTHER PASTE FILLERS; COMPOST, MANURES, ADHESIVES, FE...",goods
3,2221.0,11-08-2025,1,Agrism Traditional Way,ागरिस्म ट्रेडिशनल वे,NaN,logos/5997597.png,5997597.0,27-06-2023,Kalyani Rani,कल्याणी रानी,"Pathai Ward No-1 Rukunpur, Rukunpur urf Pathai, Sitamarhi, Bihar- 843316 India Individual",NaN,pending,Kolkata,"Organic fertilizers, Manures, Artificial manures, Manures containing antioxidants, Compost, manures, fertilizers THE MARK SHALL BE LIMITED TO THE COLOURS AS SHOWN IN THE REPRESENTATION ON THE FORM OF THE APPLICATION.. THIS IS CONDITION OF REGISTRATION THAT APPLIED MARK SHALL BE USED TOGETHER..",goods
4,2221.0,11-08-2025,1,Opta-Zan,ोप्ता-जान,NaN,logos/5997936.png,5997936.0,27-06-2023,Kmc Oiltools India Private Limited,कंस ोिलतुलस इंडिया प्राइवेट लिमिटेड,"1st Floor, Unit No. 101, Gundecha Solitare, Western Express Highway, Borivali East, Mumbai Suburban, Maharashtra, 400066 Body Incorporate",07-10-2020,registered,Mumbai,Oil drilling chemicals as covered under,goods
5,2221.0,11-08-2025,1,Micrown,मिक्रोण,NaN,logos/6003448.png,6003448.0,03-07-2023,Mr. Pankaj Nikumbh,मर. पंकज निकुम्भ,"Base No.1, Akshardham, Pethphata, Opp. Gulab Baug, Panchvati, Nashik- 422003 Sole Proprietor",01-04-2018,registered,Mumbai,"Plant Growth Regulator , Bio stimulants & Fertilizer Used In Agriculture",goods
6,2221.0,11-08-2025,1,Rxxx,रक्सक्सक्स,NaN,logos/6004472.png,6004472.0,03-07-2023,Mr. Pankaj Nikumbh,मर. पंकज निकुम्भ,"Base No.1, Akshardham, Pethphata, Opp. Gulab Baug, Panchvati, Nashik- 422003. Sole Proprietor",01-04-2018,registered,Mumbai,"Plant Growth Regulator , Bio stimulants & Fertilizer Used In Agriculture",goods
7,2221.0,11-08-2025,1,Supreme Surfacants Pit Ltd,सुप्रीम सुरफकेंट्स पित ल्टड,NaN,logos/6013748.png,6013748.0,10-07-2023,M/S Supreme Surfactants Pvt. Ltd.,म/स सुप्रीम सूर्फक्टान्ट्स पवत. ल्टड.,"405, 4TH FLOOR, PLOT NO. A-4/1, USHA KIRAN BUILDING, NANIWALA BAGH, AZADPUR, NEW DELHI-110033 A COMPANY INCORPORATED UNDER THE INDIAN COMPANIES ACT",05-02-20

Shape: (18514, 17)
Dtypes:


journal_no                       float64
registration_date                 object
nice_classes                      object
trademark_text                    object
trademark_text_devanagari         object
international_registration_no     object
image_url                         object
application_number               float64
filing_date                       object
owner_name                        object
owner_name_devanagari             object
owner_address                     object
used_since                        object
status                            object
registration_city                 object
goods_services_description        object
business_type_category            object
dtype: object

In [8]:
# CELL 4 - Column summary & missingness
cols = df.columns.tolist()
col_summary = []
for col in cols:
    s = df[col]
    non_null = s.dropna()
    col_summary.append({
        'column': col,
        'dtype': str(s.dtype),
        'n_unique': int(s.nunique(dropna=True)),
        'missing_count': int(s.isna().sum()),
        'missing_pct': float(s.isna().mean()*100),
        'sample_values': ", ".join(map(str, non_null.sample(min(5, len(non_null))).astype(str).tolist())) if len(non_null)>0 else ""
    })
col_summary_df = pd.DataFrame(col_summary).sort_values('missing_pct', ascending=False)
safe_display_df(col_summary_df, "Column summary (dtype, unique, missing%)")

missing_df = pd.concat([col_summary_df[['column','missing_count','missing_pct']].set_index('column')], axis=1)
missing_df = missing_df.reset_index()
safe_display_df(missing_df, "Missing summary")


--- Column summary (dtype, unique, missing%) ---


,column,dtype,n_unique,missing_count,missing_pct,sample_values
5,international_registration_no,object,2060,16454,88.873285,"1851587.0, 1775112.0, 1831901.0, 1809131, 1774060.0"
12,used_since,object,3322,13604,73.479529,"03/03/2023, 15/11/2021, 22-11-2022, 29/03/2019, 01-04-2022"
11,owner_address,object,15731,207,1.118073,"HIG 1/23, RAJNI KHAND, BHADRUKH, LUCKNOW - 226002, PLOT NO. 21, RADHE RESIDENCY, RAM NAGAR, VIJALPOR, NAVSARI, GUJARAT - 396450, S1, BINDAVAN BUILDING , VINAYAK MANDIR ROAD, NEAR RELIANCE OFFICE, RADHA SADAN, BHYANDER (W) - 401101, TAKECHAND JAIN BHAIJJI, 278/217, C/O, ANANDSHRI HOTAL, INFRONT SANICHARI TORLSAGAR,470002, WorkYard, Plot No. 337, Industrial Area, Phase 2, Ind Area Chandigarh, Ch..."
8,filing_date,object,2517,4,0.021605,"28/07/2023, 09-10-2023, 10/11/2023, 18-06-2024, 07/11/2022"
14,registration_city,object,6,2,0.010803,"IR DIVISION, Delhi, Delhi, Chennai, Chennai"
15,goods_services_description,object,16100,2,0.010803,"PHARMACEUTICALS AND MEDICINAL PREPARATIONS INCLUDED IN CLASS 05, Pharmaceutical preparations, namely, prothrombin complex (human), BUILDING CONSTRUCTION, REPAIR, INSTALLATION SERVICES., Education services., Pharmaceuticals, Pharmaceutical preparations, Pharmaceutical drugs, Pharmaceutical substances, Homeopathic pharmaceuticals, Antibacterial pharmaceuticals, Chemico- pharmaceutical preparatio..."
2,nice_classes,object,94,1,0.005401,"25, 3.0, 25, 30.0, 41"
0,journal_no,float64,8,0,0.000000,"2223.0, 2225.0, 2228.0, 2221.0, 2224.0"
1,registration_date,object,9,0,0.000000,"18-08-2025, 01/09/2025, 11-08-2025, 01/09/2025, 22/09/2025"
7,application_number,float64,18512,0,0.000000,"6486833.0, 5229153.0, 7069402.0, 7028512.0, 6126480.0"


--- Missing summary ---


,column,missing_count,missing_pct
0,international_registration_no,16454,88.873285
1,used_since,13604,73.479529
2,owner_address,207,1.118073
3,filing_date,4,0.021605
4,registration_city,2,0.010803
5,goods_services_description,2,0.010803
6,nice_classes,1,0.005401
7,journal_no,0,0.000000
8,registration_date,0,0.000000
9,application_number,0,0.000000


In [9]:
# CELL 5 - Null-like placeholders & mixed-type detection
null_like_report = []
for col in df.columns:
    if df[col].dtype == object or pd.api.types.is_string_dtype(df[col]):
        s = df[col].astype(str).str.strip()
        lower = s.str.lower()
        mask = lower.isin([p.lower() for p in NULL_LIKE_PATTERNS])
        cnt = int(mask.sum())
        pct = float(cnt / len(df) * 100) if len(df)>0 else 0.0
        if cnt > 0:
            null_like_report.append({'column': col, 'count_null_like': cnt, 'pct_null_like': pct})
null_like_df = pd.DataFrame(null_like_report).sort_values('count_null_like', ascending=False) if null_like_report else pd.DataFrame(columns=['column','count_null_like','pct_null_like'])
safe_display_df(null_like_df, "Null-like placeholder report")

# Mixed types (object columns with many numeric-like strings)
mixed_report = []
for col in df.columns:
    if df[col].dtype == object:
        non_null = df[col].dropna().astype(str)
        coerced = pd.to_numeric(non_null.str.replace(',',''), errors='coerce')
        n_numeric_like = int(coerced.notna().sum())
        n_nonnull = int(non_null.shape[0])
        if n_nonnull > 0 and 0 < n_numeric_like < n_nonnull:
            mixed_report.append({'column': col, 'n_non_null': n_nonnull, 'n_numeric_like': n_numeric_like, 'pct_numeric_like': float(n_numeric_like/n_nonnull*100)})
mixed_df = pd.DataFrame(mixed_report).sort_values('pct_numeric_like', ascending=False) if mixed_report else pd.DataFrame()
safe_display_df(mixed_df, "Mixed-type columns (object with numeric-like strings)")


--- Null-like placeholder report ---


,column,count_null_like,pct_null_like


--- Mixed-type columns (object with numeric-like strings) ---


,column,n_non_null,n_numeric_like,pct_numeric_like
0,nice_classes,18513,18511,99.989197
2,international_registration_no,2060,2056,99.805825
1,trademark_text,18514,13,0.070217
3,owner_name,18514,1,0.005401


In [10]:
# CELL 6 - Date parsing heuristics
date_candidates = [c for c in df.columns if any(k in c.lower() for k in ['date','time','dt','timestamp','created','updated'])]
date_report = []
parsed_dates = {}
for c in date_candidates:
    parsed = pd.to_datetime(df[c], errors='coerce', infer_datetime_format=True)
    parsed_dates[c] = parsed
    n_parsed = int(parsed.notna().sum())
    pct_parsed = float(n_parsed/len(df)*100) if len(df)>0 else 0.0
    date_report.append({'column': c, 'n_parsed': n_parsed, 'pct_parsed': pct_parsed})
parsed_dates_df = pd.DataFrame(date_report).sort_values('pct_parsed', ascending=False) if date_report else pd.DataFrame()
safe_display_df(parsed_dates_df, "Date parsing heuristic report")


--- Date parsing heuristic report ---


C:\Users\sharm\AppData\Local\Temp\ipykernel_22888\3521353728.py:6: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(df[c], errors='coerce', infer_datetime_format=True)
C:\Users\sharm\AppData\Local\Temp\ipykernel_22888\3521353728.py:6: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(df[c], errors='coerce', infer_datetime_format=True)
C:\Users\sharm\AppData\Local\Temp\ipykernel_22888\3521353728.py:6: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` o

,column,n_parsed,pct_parsed
1,filing_date,6915,37.350113
0,registration_date,4919,26.569083


In [11]:
# CELL 7 - Numeric diagnostics and save hist/boxplot images
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
num_summary = []
for c in numeric_cols:
    s = df[c].dropna()
    if s.shape[0] == 0:
        continue
    desc = s.describe()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_outliers = int(((s < lower) | (s > upper)).sum())
    skewness = float(s.skew()) if hasattr(s, 'skew') else None
    num_summary.append({
        'column': c, 'count': int(desc['count']), 'mean': float(desc['mean']), 'std': float(desc['std']),
        'min': float(desc['min']), '25%': float(q1), '50%': float(desc['50%']), '75%': float(q3), 'max': float(desc['max']),
        'n_outliers_IQR': n_outliers, 'skewness': skewness
    })

numeric_summary_df = pd.DataFrame(num_summary).sort_values('n_outliers_IQR', ascending=False) if num_summary else pd.DataFrame()
safe_display_df(numeric_summary_df, "Numeric diagnostics")

# save histogram + boxplot for each numeric column
plot_index = []  # collect metadata for HTML
for c in numeric_cols:
    s = df[c].dropna()
    if s.shape[0] < 2:
        continue
    # histogram
    fig, ax = plt.subplots(figsize=(8,3))
    ax.hist(s, bins=30)
    ax.set_title(f'Histogram: {c}')
    ax.set_xlabel(c); ax.set_ylabel('count')
    hist_path = save_fig(fig, f"{c}_hist.png")
    plot_index.append({'file': hist_path.name,'title': f'Histogram: {c}','desc': f'Distribution of {c} (n={s.shape[0]})'})
    # boxplot
    fig, ax = plt.subplots(figsize=(6,2))
    ax.boxplot(s, vert=False)
    ax.set_title(f'Boxplot: {c}')
    box_path = save_fig(fig, f"{c}_box.png")
    plot_index.append({'file': box_path.name,'title': f'Boxplot: {c}','desc': f'Boxplot (IQR) for {c}'})


--- Numeric diagnostics ---


,column,count,mean,std,min,25%,50%,75%,max,n_outliers_IQR,skewness
1,application_number,18514,6.210278e+06,515237.913248,1150043.0,6021944.5,6179548.5,6485906.0,7238449.0,911,-1.830224
0,journal_no,18514,2.224222e+03,2.339103,2221.0,2222.0,2224.0,2226.0,2228.0,0,0.160250


In [12]:
# CELL 8 - Categorical diagnostics and bar charts for top values
cat_cols = [c for c in df.columns if (df[c].dtype == object or pd.api.types.is_categorical_dtype(df[c])) and c not in numeric_cols]
cat_report = []
for c in cat_cols:
    s = df[c].astype(str)
    n_unique = int(s.nunique(dropna=True))
    top_values = s.value_counts(dropna=True).head(10).to_dict()
    n_strip_issues = int((s != s.str.strip()).sum())
    lower_unique = s.str.lower().nunique()
    stripped_lower_unique = s.str.strip().str.lower().nunique()
    n_case_variants = int(lower_unique - stripped_lower_unique)
    cat_report.append({'column': c, 'n_unique': n_unique, 'n_strip_issues': n_strip_issues, 'n_case_variants': n_case_variants, 'top_values_sample': str(top_values)[:1000]})

cat_report_df = pd.DataFrame(cat_report).sort_values('n_unique', ascending=False) if cat_report else pd.DataFrame()
safe_display_df(cat_report_df, "Categorical diagnostics")

# Barplots for top categorical columns (limit to avoid huge output)
top_cat_cols = cat_report_df.sort_values('n_unique', ascending=False).head(6)['column'].tolist() if not cat_report_df.empty else cat_cols[:6]
for col in top_cat_cols:
    vc = df[col].astype(str).value_counts(dropna=False).head(20)
    if vc.empty: 
        continue
    fig, ax = plt.subplots(figsize=(8,4))
    ax.barh(vc.index.astype(str)[::-1], vc.values[::-1])
    ax.set_title(f'Top values — {col}')
    ax.set_xlabel('Count')
    cat_path = save_fig(fig, f"{col}_top_values.png")
    plot_index.append({'file': cat_path.name, 'title': f'Top values: {col}', 'desc': f'Top values for {col} (top 20).'})


C:\Users\sharm\AppData\Local\Temp\ipykernel_22888\2484213572.py:2: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  cat_cols = [c for c in df.columns if (df[c].dtype == object or pd.api.types.is_categorical_dtype(df[c])) and c not in numeric_cols]


--- Categorical diagnostics ---


,column,n_unique,n_strip_issues,n_case_variants,top_values_sample
5,image_url,18512,0,0,"{'logos/5937368.png': 2, 'logos/6185736.png': 2, 'logos/5677121.png': 1, 'logos/5678428.png': 1, 'logos/5900889.png': 1, 'logos/6278318.png': 1, 'logos/6702858.png': 1, 'logos/6775066.png': 1, 'logos/6801915.png': 1, 'logos/6803662.png': 1}"
2,trademark_text,16843,64,6,"{'Nan': 392, 'Me Tiles\nInfinite Surfaces': 17, 'Zcd': 13, 'Clew': 11, 'Neo Swift Unicoat': 10, 'Sri': 10, 'M': 9, 'For The Love Of Pizza': 9, 'Reid & Taylor': 8, 'Skibidi Toilet': 8}"
3,trademark_text_devanagari,16826,73,6,"{'नान': 392, 'में टाइल्स\nइनफिनिट सर्फेस': 17, 'जकड़': 14, 'बनारस भांग': 13, 'क्लेव': 11, 'श्री': 10, 'फॉर थे लोवे ऑफ़ पिज़्ज़ा': 9, 'सागर': 9, 'म': 9, 'फॉर थे लोवे ऑफ़ पॉपकॉर्न': 8}"
13,goods_services_description,16101,2,0,"{'Pharmaceutical and medicinal preparations and substances; vaccines.': 67, 'Pharmaceuticals, medical and veterinary preparations; Sanitary preparations for medical purposes; Dietetic food and substances adapted for medical or veterinary use, food for babies; Dietary supplements for human beings and animals; Plasters, materials for dressings; Material for stopping teeth, dental wax; Disinfecta..."
9,owner_address,15732,3,0,"{'nan': 207, 'GSK Medicines Research Centre Gunnels Wood Road, Stevenage SG1 2NY GB': 39, '65 Railroad Avenue, Suite 101-D, Ridgefield, NJ 07657 USA A New Jersey Corporation': 24, '5-6, Sheetal Apartment, Ground Floor, Opp. Chandan Cinema, Juhu, Mumbai 400049': 17, '101, RADHKA, 55 GULMOHAR ROAD, JUHU SCHEME, VILE PARLE (WEST), MUMBAI-400049, MAHARASHTRA, INDIA': 15, 'CH-4002 Basel Switzerland..."
8,owner_name_devanagari,15135,18,0,"{'ग्लाक्सो ग्रुप लिमिटेड': 66, 'पसटगेम्स प्राइवेट लिमिटेड': 28, 'ेक्सो ग्लोबल प्राइवेट लिमिटेड': 21, 'सत्यराम सिल्क मिल्स लिमिटेड': 19, 'न्य सिनेमाज ललप': 17, 'थे फ़ीनिक्स मिल्स लिमिटेड': 14, 'स.र.ी. इम्पेक्स प्राइवेट लिमिटेड': 14, 'अल्केम लैबोरेट्रीज ल्टड.': 14, 'नोवार्टिस एजी': 14, 'थे साउथ इंडियन बैंक लिमिटेड': 14}"
7,owner_name,15006,3,0,"{'Glaxo Group Limited': 66, 'Pstgems Private Limited': 28, 'Ecso Global Private Limited': 24, 'Styaram Silk Mills Limited': 19, 'Ny Cinemas Llp': 17, 'M/S. Neospark Drugs And Chemicals Private Limited': 17, 'Nippon Paint (India) Pvt. Ltd': 15, 'S.R.I. Impex Private Limited': 15, 'Grasim Industries Limited': 14, 'Novartis Ag': 14}"
10,used_since,3323,0,0,"{'nan': 13604, '01/04/2023': 20, '01-04-2023': 20, '01/04/2021': 17, '01-04-2022': 15, '01/07/2023': 13, '01/04/2017': 12, '01-01-2023': 12, '01/08/2023': 11, '01-05-2023': 11}"
6,filing_date,2518,0,0,"{'20/06/2024': 418, '21/06/2024': 357, '19/06/2024': 309, '18-06-2024': 281, '18/06/2024': 223, '19-06-2024': 119, '17-06-2024': 77, '02/12/2023': 62, '17/06/2024': 60, '08/11/2023': 59}"
4,international_registration_no,2061,0,0,"{'nan': 16454, '1762871': 1, '1455089': 1, '1813035': 1, '1813274': 1, '1813273': 1, '1813230': 1, '1259588': 1, '1817391': 1, '1819520': 1}"


C:\Users\sharm\AppData\Local\Temp\ipykernel_22888\1279852874.py:16: UserWarning: Glyph 2337 (\N{DEVANAGARI LETTER DDA}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\sharm\AppData\Local\Temp\ipykernel_22888\1279852874.py:16: UserWarning: Matplotlib currently does not support Devanagari natively.
  fig.tight_layout()
C:\Users\sharm\AppData\Local\Temp\ipykernel_22888\1279852874.py:16: UserWarning: Glyph 2375 (\N{DEVANAGARI VOWEL SIGN E}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\sharm\AppData\Local\Temp\ipykernel_22888\1279852874.py:16: UserWarning: Glyph 2354 (\N{DEVANAGARI LETTER LA}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\sharm\AppData\Local\Temp\ipykernel_22888\1279852874.py:16: UserWarning: Glyph 2368 (\N{DEVANAGARI VOWEL SIGN II}) missing from font(s) DejaVu Sans.
  fig.tight_layout()
C:\Users\sharm\AppData\Local\Temp\ipykernel_22888\1279852874.py:16: UserWarning: Glyph 2327 (\N{DEVANAGARI LETTER GA}) missing from fon

In [13]:
# CELL 9 - Correlation heatmap and pair scatter sample plots
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(8,6))
    cax = ax.matshow(corr, vmin=-1, vmax=1)
    fig.colorbar(cax)
    ax.set_xticks(range(len(numeric_cols))); ax.set_yticks(range(len(numeric_cols)))
    ax.set_xticklabels(numeric_cols, rotation=90); ax.set_yticklabels(numeric_cols)
    corr_path = save_fig(fig, "correlation_matrix.png")
    plot_index.append({'file': corr_path.name, 'title': 'Correlation matrix', 'desc': 'Pearson correlations for numeric columns'})

    # scatterplots for up to first 4 numeric columns pairwise
    from itertools import combinations
    pair_cols = numeric_cols[:4]
    for a,b in combinations(pair_cols, 2):
        s = df[[a,b]].dropna()
        if s.shape[0] < 10:
            continue
        fig, ax = plt.subplots(figsize=(4,3))
        ax.scatter(s[a], s[b], s=6, alpha=0.6)
        ax.set_xlabel(a); ax.set_ylabel(b)
        ax.set_title(f'Scatter: {a} vs {b}')
        spath = save_fig(fig, f"scatter_{a}_vs_{b}.png")
        plot_index.append({'file': spath.name, 'title': f'Scatter: {a} vs {b}', 'desc': f'Scatter of {a} vs {b} (n={s.shape[0]})'})
else:
    print("Not enough numeric columns for correlation matrix.")


In [14]:
# CELL 10 - Consolidated issues (recomputed) and recommendations
issues = []
# high missingness
for _, row in col_summary_df.iterrows():
    if row['missing_pct'] > 50:
        issues.append({'column': row['column'], 'issue': 'High missingness', 'detail': f"{row['missing_pct']:.2f}% missing"})
# duplicates
dup_count = int(df.duplicated(keep=False).sum())
if dup_count > 0:
    issues.append({'column': 'ALL', 'issue': 'Duplicate rows', 'detail': f"{dup_count} exact duplicate rows"})
# null-like placeholders
for _, r in null_like_df.iterrows():
    issues.append({'column': r['column'], 'issue': 'Null-like placeholders', 'detail': f"{int(r['count_null_like'])} values like NA/N/A/none etc"})
# mixed types
for _, r in mixed_df.iterrows():
    issues.append({'column': r['column'], 'issue': 'Mixed types', 'detail': f"{int(r['n_numeric_like'])}/{int(r['n_non_null'])} numeric-like strings"})
# date parsing issues
for _, r in parsed_dates_df.iterrows():
    if r['pct_parsed'] < 70:
        issues.append({'column': r['column'], 'issue': 'Date parse failure/ambiguous', 'detail': f"{r['pct_parsed']:.2f}% parsed"})
# high-cardinality categorical
for _, r in cat_report_df.iterrows():
    if r['n_unique'] > 100:
        issues.append({'column': r['column'], 'issue': 'High cardinality categorical', 'detail': f"{r['n_unique']} unique values"})
# numeric outliers
for _, r in numeric_summary_df.iterrows():
    if r['n_outliers_IQR'] > 0:
        issues.append({'column': r['column'], 'issue': 'Potential outliers', 'detail': f"{r['n_outliers_IQR']} outliers (IQR)"})

issues_df = pd.DataFrame(issues)
safe_display_df(issues_df, "Consolidated issues")

# Recommendations (automated suggestions)
recommendations = [
    "Drop columns with >80% missing or fill with domain-specific defaults.",
    "Normalize categorical columns (strip whitespace, lower-case) and re-evaluate unique counts.",
    "Coerce mixed-type columns to numeric after cleaning (remove commas, currency signs).",
    "Inspect date columns with low parse rate and implement custom parsing.",
    "Investigate rows flagged as duplicates and either deduplicate or keep with reason.",
    "For numeric outliers, confirm business rules before trimming or capping."
]


--- Consolidated issues ---


,column,issue,detail
0,international_registration_no,High missingness,88.87% missing
1,used_since,High missingness,73.48% missing
2,ALL,Duplicate rows,4 exact duplicate rows
3,nice_classes,Mixed types,18511/18513 numeric-like strings
4,international_registration_no,Mixed types,2056/2060 numeric-like strings
5,trademark_text,Mixed types,13/18514 numeric-like strings
6,owner_name,Mixed types,1/18514 numeric-like strings
7,filing_date,Date parse failure/ambiguous,37.35% parsed
8,registration_date,Date parse failure/ambiguous,26.57% parsed
9,image_url,High cardinality categorical,18512 unique values


In [15]:
# CELL 11 - Generate full HTML report that includes tables, plots and recommendations
html_parts = []
html_parts.append(f"<html><head><meta charset='utf-8'><title>EDA Report</title>")
# basic styles
html_parts.append("""<style>
body{font-family:Arial,Helvetica,sans-serif;margin:30px;color:#111}
h1,h2,h3{color:#222}
table.table{border-collapse:collapse;width:100%}
table.table th, table.table td {border:1px solid #ddd;padding:6px}
</style></head><body>""")

html_parts.append(f"<h1>EDA Report — generated {timestamp()}</h1>")
html_parts.append("<p>This report contains schema, missingness, null-like placeholders, mixed-type detections, date parsing results, numeric & categorical diagnostics, plots, issues and recommendations.</p>")

# Column summary
html_parts.append(df_to_html_table(col_summary_df, "Column summary (dtype, unique, missing%)", max_rows=200))

# Missing
html_parts.append(df_to_html_table(missing_df, "Missing summary", max_rows=200))

# Null-like and mixed-type
html_parts.append(df_to_html_table(null_like_df, "Null-like placeholder report", max_rows=200))
html_parts.append(df_to_html_table(mixed_df, "Mixed-type object columns", max_rows=200))

# Date parsing
html_parts.append(df_to_html_table(parsed_dates_df, "Date parsing heuristic report", max_rows=200))

# Numeric summary
html_parts.append(df_to_html_table(numeric_summary_df, "Numeric diagnostics summary", max_rows=200))

# Categorical
html_parts.append(df_to_html_table(cat_report_df, "Categorical diagnostics", max_rows=200))

# Issues
html_parts.append(df_to_html_table(issues_df, "Consolidated issues (auto-detected)", max_rows=500))

# Recommendations
html_parts.append("<h2>Automated recommendations</h2><ul>")
for rec in recommendations:
    html_parts.append(f"<li>{rec}</li>")
html_parts.append("</ul>")

# Plots gallery (list plot_index)
html_parts.append("<h2>Plots gallery</h2>")
if len(plot_index)==0:
    html_parts.append("<p>No plots were generated.</p>")
else:
    for it in plot_index:
        img_rel = "plots/" + it['file']
        html_parts.append(f"<div style='margin-bottom:20px'><h3>{it['title']}</h3>")
        html_parts.append(f"<a href='{img_rel}' target='_blank'><img src='{img_rel}' style='max-width:900px;border:1px solid #ccc;margin-bottom:6px'></a>")
        html_parts.append(f"<p>{it['desc']}</p></div><hr/>")

# Footer and end
html_parts.append(f"<p>Files saved in folder: {OUT_DIR}</p>")
html_parts.append("</body></html>")

HTML_CONTENT = "\n".join(html_parts)
with open(HTML_REPORT_PATH, "w", encoding="utf-8") as f:
    f.write(HTML_CONTENT)
print("Saved HTML report to:", HTML_REPORT_PATH)
# show quick inline link
display(HTML(f"<a href='file://{HTML_REPORT_PATH}' target='_blank'>Open EDA report: {HTML_REPORT_PATH.name}</a>"))


Saved HTML report to: C:\Users\sharm\Desktop\Bhanu\GitHub\TMParseOCR\eda_reports\EDA_report.html


In [16]:
# CELL 12 - Save CSVs and zip outputs
col_summary_df.to_csv(OUT_DIR / 'column_summary.csv', index=False)
missing_df.to_csv(OUT_DIR / 'missing_summary.csv', index=False)
null_like_df.to_csv(OUT_DIR / 'null_like_report.csv', index=False)
mixed_df.to_csv(OUT_DIR / 'mixed_report.csv', index=False)
parsed_dates_df.to_csv(OUT_DIR / 'parsed_dates_report.csv', index=False)
numeric_summary_df.to_csv(OUT_DIR / 'numeric_summary.csv', index=False)
cat_report_df.to_csv(OUT_DIR / 'categorical_report.csv', index=False)
issues_df.to_csv(OUT_DIR / 'issues_report.csv', index=False)

from zipfile import ZipFile, ZIP_DEFLATED

# Put the zip file inside the same OUT_DIR
ZIP_PATH = OUT_DIR / "EDA_reports.zip"

# If an old zip exists, remove it first to avoid including it in the new zip
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# Create the zip inside OUT_DIR; use arcname relative to OUT_DIR so internal paths are neat
with ZipFile(ZIP_PATH, 'w', compression=ZIP_DEFLATED) as zf:
    for file in OUT_DIR.rglob('*'):
        # skip the zip file itself (important)
        if file == ZIP_PATH:
            continue
        # write file with path relative to OUT_DIR so zip has a clean internal structure
        zf.write(file, arcname=file.relative_to(OUT_DIR))
print("Created zip inside output dir:", ZIP_PATH)



Created zip inside output dir: C:\Users\sharm\Desktop\Bhanu\GitHub\TMParseOCR\eda_reports\EDA_reports.zip
